## Variance partitioning of TF sample loading using metadata

In [ ]:
library(ggsci)
library(lme4)
library(variancePartition)
library(dplyr)
library(tidyverse)

options(stringsAsFactors=F)
options(scipen=100)

In [ ]:
# ---- Set the following paths to match your environment ----
wd_path  = "/path/to/shell_script"      # directory of shell script
TF_metadata_file = "/path/to/TF_ChIPseq.info"  # directory of TF metadata
result_path  = "/path/to/result"            # directory for CATaN output
save_path  = "/path/to/output"            # directory for output
# -----------------------------------------------------------

CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

anno_df=read.table(TF_metadata_file, header=1)
colnames(anno_df)=c("TFpname", "TFgname", "GSM", "Cell", "Cell_CA")


all_variance_df=data.frame(matrix(rep(NA, 3), nrow=1))[numeric(0), ]
CC_path=paste0(result_path,"/raw_CC") ##TF sample loading
for (i in seq(10)){
    CC_df=read.table(paste0(CC_path, "/", CC_list[i], "_CCscore.txt"))
    colnames(CC_df)=c("id", "CCscore")

    CC_df1=CC_df %>% separate(id, c("GSM", "TF"), sep="_")
    rownames(CC_df1)=CC_df$id
    merge_df=left_join(CC_df1, anno_df, by="GSM")
  

    ML  = lmer(CCscore	 ~ (1|TFpname)+(1|Cell), merge_df, REML = FALSE, verbose = FALSE, na.action = na.omit)
    Var_Random_effect = unlist(VarCorr(ML))
    Var_Residual = attr(VarCorr(ML), "sc")^2
    Var_Ramdom   = c(Var_Random_effect,Var_Residual)
        
    Var_all=Var_Ramdom
    Sum_all   = sum(Var_all)
    Var_all_std = data.frame(Var_all/Sum_all) %>%t() %>% as.data.frame()
    colnames(Var_all_std) = c("TFpname","tissue","residual")
    rownames(Var_all_std) = paste0("CC", i)
    all_variance_df=rbind(all_variance_df, Var_all_std)
}
all_variance_df$id=rownames(all_variance_df)
all_variance_df2=all_variance_df %>% pivot_longer(cols=c("TFpname", "tissue", "residual"))
all_variance_df2$name=factor(all_variance_df2$name, levels=c("residual", "tissue", "TFpname"))
all_variance_df2$id=factor(all_variance_df2$id,levels=c("CC1", "CC2", "CC3", "CC4", "CC5", "CC6", "CC7", "CC8", "CC9", "CC10"))
g <- ggplot(all_variance_df2, aes(x = id, y = value, fill = name))
g <- g + geom_bar(stat = "identity")
g <- g + scale_fill_nejm()
ggsave(paste0(save_path, "/Variance_partitioning_TF.png"), width=6, height=8)
write.table(all_variance_df2, paste0(save_path,"/variacne_partitioning.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)


sessionInfo()


R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.8 (Ootpa)

Matrix products: default
BLAS/LAPACK: /hshare1/ZETTAI_path_WA_slash_home_KARA/home/imgtaka/miniconda3/envs/renv/lib/libopenblasp-r0.3.26.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=ja_JP.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=ja_JP.UTF-8        LC_COLLATE=ja_JP.UTF-8    
 [5] LC_MONETARY=ja_JP.UTF-8    LC_MESSAGES=ja_JP.UTF-8   
 [7] LC_PAPER=ja_JP.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=ja_JP.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Tokyo
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.3          forcats_1.0.0            stringr_1.5.1           
 [4] purrr_1.0.2              readr_2.1.5              tidyr_1.3.1             
 [7] tibble_3.2.1             tidyverse_2.0.0          dplyr_1.1.4             
[10] variancePartition_1.32.5 BiocParallel_1.36.0      limma_3.58.1            
[13] ggplot2_3.5.1            lme4_1.1-35.3            Matrix_1.6-5            
[16] ggsci_3.0.3             

loaded via a namespace (and not attached):
 [1] tidyselect_1.2.1    IRdisplay_1.1       bitops_1.0-9       
 [4] fastmap_1.2.0       digest_0.6.37       timechange_0.3.0   
 [7] lifecycle_1.0.4     statmod_1.5.0       magrittr_2.0.3     
[10] compiler_4.3.3      rlang_1.1.4         tools_4.3.3        
[13] plyr_1.8.9          repr_1.1.7          KernSmooth_2.23-22 
[16] pbdZMQ_0.3-11       withr_3.0.2         numDeriv_2016.8-1.1
[19] BiocGenerics_0.48.1 grid_4.3.3          aod_1.3.3          
[22] caTools_1.18.3      colorspace_2.1-1    scales_1.3.0       
[25] gtools_3.9.5        iterators_1.0.14    MASS_7.3-60        
[28] cli_3.6.3           mvtnorm_1.2-4       crayon_1.5.3       
[31] generics_0.1.3      reshape2_1.4.4      tzdb_0.4.0         
[34] minqa_1.2.6         splines_4.3.3       parallel_4.3.3     
[37] matrixStats_1.5.0   base64enc_0.1-3     vctrs_0.6.5        
[40] boot_1.3-31         jsonlite_1.8.9      hms_1.1.3          
[43] pbkrtest_0.5.2      glue_1.8.0          nloptr_2.0.3       
[46] codetools_0.2-19    stringi_1.8.4       gtable_0.3.6       
[49] EnvStats_2.8.1      lmerTest_3.1-3      munsell_0.5.1      
[52] remaCor_0.0.18      pillar_1.10.1       htmltools_0.5.8.1  
[55] gplots_3.2.0        IRkernel_1.3.2      R6_2.5.1           
[58] Rdpack_2.6          evaluate_1.0.3      lattice_0.22-6     
[61] Biobase_2.62.0      rbibutils_2.2.16    backports_1.4.1    
[64] RhpcBLASctl_0.23-42 broom_1.0.5         fANCOVA_0.6-1      
[67] corpcor_1.6.10      Rcpp_1.0.14         uuid_1.2-1         
[70] nlme_3.1-164        pkgconfig_2.0.3    

Click to add a cell.